> **Sobre las cifras.** Lo que este notebook imprime al ejecutarse es ilustrativo. Las cifras citables del proyecto son las de `resultados/RESULTADOS.md` y `resultados/resultados_congelados.json` (las produce `06_congelar_resultados.ipynb`). La última celda de este notebook compara esta corrida con esos valores.

In [ ]:
# Preparación (no modificar): ubica la raíz del repo, la usa como directorio de trabajo
# y pone codigo/ en el path. Funciona igual abriendo el notebook desde notebooks/ o desde
# la raíz, y también ejecutándolo con codigo/run_nb.py.
import os, sys
_raiz = os.getcwd()
while not os.path.isdir(os.path.join(_raiz, "codigo")) and os.path.dirname(_raiz) != _raiz:
    _raiz = os.path.dirname(_raiz)
os.chdir(_raiz)
sys.path.insert(0, os.path.join(_raiz, "codigo"))
print("Raíz del repo:", _raiz)

## Configuración: SIN trim en ambos años (decisión del 2026-09-17)

Para que la comparación sea válida, los dos años corren bajo el **mismo tratamiento**: sin
recorte iterativo y con `ing_cor` como denominador (la única variable de ingreso con
definición idéntica en 2014 y 2022).

Las cifras de bienestar de aquí **no coinciden** con las de `01_replica_2014.ipynb`, que es la
réplica de fidelidad al Gauss (con trim) y sirve de benchmark contra el paper. Ver
`resultados/RESULTADOS.md`.

# Comparación 2014 ↔ 2022

Corre los dos años bajo **tratamiento idéntico** para que las diferencias observadas sean
atribuibles al cambio en la economía y no a la configuración del estimador.


> **Actualizado (5 ago 2026):** incorpora las cuatro correcciones de fidelidad al Gauss en
> la ruta de bienestar — V1 umbral de significancia t ≥ 2.326 (l.6376), V2 markup de Lerner
> −1/ε para el contrafactual (l.6345, 6380), V3 denominador `ing_cor`, V4 medianas en vez de
> medias (l.6520). V2 y V4 **interactúan**: evaluados por separado dan una lectura invertida.

---

## Por qué existe este notebook

`01_replica_2014.ipynb` y `02_actualizacion_2022.ipynb` corren cada año en su **mejor**
configuración, que no es la misma:

* **2014 con trim** — es la réplica de fidelidad al Gauss: el recorte del 1 % por cola cuesta el 28 %
  de la muestra (12,372 → 8,940) y sí cambia los resultados de bienestar (P2 §5).
* **2022 sin trim** — con 57,552 hogares ese mismo recorte colapsa la varianza del regresor
  de utilidad y deja al 65 % de los hogares con efectos ingreso planos (`trim_2022`).

Esa asimetría es correcta para reportar **niveles** de cada año, pero inutiliza cualquier
afirmación del tipo *"el impuesto subió de X a Y"*: la diferencia confundiría el cambio
económico con el cambio de tratamiento.

**Aquí se corren ambos años sin trim y con el mismo denominador de ingreso.**

### Las dos decisiones de comparabilidad

**Trim: desactivado en ambos.** Es el único ajuste que permite tratar los dos años igual.
Tiene un costo, y hay que declararlo: sin trim, 2014 ya no es la réplica de fidelidad al Gauss.
Se acepta porque el objetivo aquí es comparar, no replicar.

**Ingreso: `ing_cor` en ambos.** Es la única variable de ingreso con definición idéntica en
los dos concentrados. `ing_mon` no sirve para comparar: la ENIGH 2022 "Nueva serie" dejó de
publicarlo y hubo que reconstruirlo, así que no se puede saber si la brecha entre años es cambio real
o diferencia de concepto.

**Lo que este notebook NO reemplaza:** el Gini de 2014 sobre `ing_mon` (0.481, idéntico al
publicado) sigue siendo el resultado de la réplica, y vive en `01_replica_2014.ipynb`.

---

### Advertencia sobre magnitudes en pesos

Las magnitudes en pesos no están validadas (falta el ajuste a pesos constantes). **Usar
porcentajes**; el nivel de la pérdida depende de la especificación (P2 §5).

In [ ]:
import sys
sys.path.insert(0, "codigo")
import numpy as np

import datos_2014, datos_2022
from aradillas_core import (estimar_easi, reconstruir_matrices, ModeloEASI,
                            demandas_marshallianas, elasticidades,
                            estimar_markups, variacion_equivalente,
                            cuadro_10, gini, sectores_significativos)


def correr(mod, ruta, etiqueta):
    """Pipeline completo bajo tratamiento comparable: sin trim, ingreso corriente."""
    d = mod.cargar(ruta, verbose=False)
    r = estimar_easi(d.precios_ln, d.w, d.gasto_total, d.Z, n_cat=d.n_cat,
                     aplicar_trim=False, verbose=False)
    d = d.submuestra(r["mask"])
    modelo = ModeloEASI(**reconstruir_matrices(r["beta"], n_cat=d.n_cat))
    util = modelo.utilidad_indirecta(d.precios_ln, d.Z, r["epsilon"], d.w,
                                     d.gasto_total, verbose=False)
    dem, _ = demandas_marshallianas(modelo, d.precios_ln, d.Z, util, r["epsilon"],
                                    d.gasto_total, d.factor_expansion, d.n_cat)
    e_nac, e_cd = elasticidades(modelo, d.precios_ln, d.Z, r["epsilon"], d.w,
                                d.gasto_total, d.factor_expansion, dem, d.ciudad,
                                d.n_ciudades, d.n_cat, util,
                                nombres=d.nombres_cat, verbose=False)
    mk = estimar_markups(d.precios_por_ciudad(), e_cd, d.vars_costos)
    sig = sectores_significativos(mk["t_eta"], mk["beta_eta"])   # V1: t >= 2.326
    VE = variacion_equivalente(modelo, d.precios_ln, d.Z, r["epsilon"], d.w,
                               d.gasto_total, mk["markup_lerner"][d.ciudad], sig,
                               verbose=False)                      # V2: Lerner
    c10 = cuadro_10(VE, d.ingreso_cor)          # V3 ing_cor + V4 mediana
    g = gini(d.ingreso_cor_completo, c10["tasas"])
    print(f"{etiqueta}: {d.n_hogares} hogares, {int(sig.sum())}/{d.n_cat} sectores signif.")
    return dict(d=d, e=np.abs(e_nac), mk=mk, sig=sig, c10=c10, g=g)


r14 = correr(datos_2014, "datos/Data_2014", "2014")
r22 = correr(datos_2022, "datos/Data_2022/", "2022")

## Elasticidades

In [ ]:
# Se empareja por NOMBRE: 2022 tiene categorías que 2014 no (Pan de caja, P1),
# así que la posición j ya no identifica la misma categoría en ambos años.
cats = r14["d"].nombres_cat
j22 = {n: k for k, n in enumerate(r22["d"].nombres_cat)}
solo22 = [n for n in r22["d"].nombres_cat if n not in cats]
print(f"{'Categoría':<22} {'2014':>8} {'2022':>8} {'cambio':>9}")
print("-"*50)
for j, n in enumerate(cats):
    a, b = r14["e"][j], r22["e"][j22[n]]
    print(f"  {n:<20} {a:>8.3f} {b:>8.3f} {b-a:>+9.3f}")
e22_comunes = np.array([r22["e"][j22[n]] for n in cats])
print(f"\n{'promedio (comunes)':<22} {r14['e'].mean():>8.3f} {e22_comunes.mean():>8.3f}"
      f" {e22_comunes.mean()-r14['e'].mean():>+9.3f}")
for n in solo22:
    print(f"  {n:<20} {'—':>8} {r22['e'][j22[n]]:>8.3f}   (solo 2022)")

## Poder de mercado (β_η)

Un β_η mayor indica mayor capacidad de fijar precio por encima del costo marginal.

In [ ]:
print(f"{'Categoría':<22} {'β 2014':>8} {'t':>7} {'β 2022':>8} {'t':>7} {'cambio':>9}")
print("-"*66)
for j, n in enumerate(cats):
    k = j22[n]
    b14, t14 = r14["mk"]["beta_eta"][j], r14["mk"]["t_eta"][j]
    b22, t22 = r22["mk"]["beta_eta"][k], r22["mk"]["t_eta"][k]
    marca = ""
    if r22["sig"][k] and not r14["sig"][j]:
        marca = "  ← gana significancia"
    elif r14["sig"][j] and not r22["sig"][k]:
        marca = "  ← pierde significancia"
    print(f"  {n:<20} {b14:>8.3f} {t14:>7.2f} {b22:>8.3f} {t22:>7.2f} {b22-b14:>+9.3f}{marca}")
for n in solo22:
    k = j22[n]
    print(f"  {n:<20} {'—':>8} {'—':>7} {r22['mk']['beta_eta'][k]:>8.3f}"
          f" {r22['mk']['t_eta'][k]:>7.2f}   (solo 2022)")
print(f"\nsectores significativos: {int(r14['sig'].sum())} en 2014, "
      f"{int(r22['sig'].sum())} en 2022")

## Bienestar: incidencia por decil

Ambos años sobre `ing_cor`, así que las cifras son directamente comparables.

In [ ]:
print(f"{'Decil':<7} {'2014 %':>9} {'2022 %':>9} {'cambio':>9}")
print("-"*38)
for f14, f22 in zip(r14["c10"]["deciles"], r22["c10"]["deciles"]):
    print(f"  {f14['decil']:<5} {f14['pct']:>9.1f} {f22['pct']:>9.1f}"
          f" {f22['pct']-f14['pct']:>+9.1f}")
t14, t22 = r14["c10"]["total"]["pct"], r22["c10"]["total"]["pct"]
print(f"  {'Tot':<5} {t14:>9.1f} {t22:>9.1f} {t22-t14:>+9.1f}")
print(f"\nRegresividad D1/D10:  2014 = {r14['c10']['regresividad']:.2f}"
      f"   2022 = {r22['c10']['regresividad']:.2f}")
print(f"\nGini observado:       2014 = {r14['g']['observado']:.3f}"
      f"   2022 = {r22['g']['observado']:.3f}")
print(f"Gini contrafactual:   2014 = {r14['g']['contrafactual']:.3f}"
      f"   2022 = {r22['g']['contrafactual']:.3f}")
print(f"Reducción del Gini:   2014 = {r14['g']['reduccion_pct']:.1f}%"
      f"   2022 = {r22['g']['reduccion_pct']:.1f}%")

## Lectura

Las diferencias de esta tabla son atribuibles al cambio en la economía entre 2014 y 2022,
**no** a la configuración del estimador — que es idéntica en ambas corridas.

Al interpretar, tener presente:

* La ENIGH 2022 tiene 90,102 hogares contra 19,124 en 2014. Las tasas de retención tras los
  filtros son casi iguales (64.4 % y 65.8 %), así que la muestra es comparable en
  composición, pero los errores estándar de 2022 son mecánicamente menores.
* Sin trim, el 36.8 % de los hogares de 2022 conserva efectos ingreso planos
  (contra 7.9 % en 2014, también sin trim). Las categorías con más desviación —Bebidas y Transporte
  foráneo— son candidatas a estar afectadas por eso y **no deberían reportarse como cambio
  económico sin verificación adicional**.
* Las magnitudes en pesos no están validadas; usar porcentajes.

In [ ]:
# Cotejo con el congelado: esta corrida contra resultados/resultados_congelados.json
import json
_ref = json.load(open("resultados/resultados_congelados.json"))
_a = _ref["comparable_2014"]["markups"]["ciudad"]["variantes_ve"]["gauss"]
_b = _ref["comparable_2022"]["markups"]["ciudad"]["variantes_ve"]["gauss"]
def _cotejo(filas, tol):
    print(f"{'cifra':<28}{'esta corrida':>14}{'congelado':>12}")
    ok = True
    for nombre, mio, cong in filas:
        bien = abs(mio - cong) <= tol[nombre]
        ok &= bien
        print(f"{nombre:<28}{mio:>14.4f}{cong:>12.4f}  {'✓' if bien else '✗ DIFERENTE'}")
    print("\nTodo coincide con el congelado." if ok else
          "\nHay diferencias: revisar versiones (numpy 1.26.4, pandas 2.2.3, scipy 1.13.1) y los datos.")
_cotejo([("2014 VE / ingreso (%)", r14["c10"]["total"]["pct"], _a["ve_pct_total"]),
         ("2014 regresividad D1/D10", r14["c10"]["regresividad"], _a["regresividad"]),
         ("2014 reducción del Gini (%)", r14["g"]["reduccion_pct"], _a["gini_ing_cor"]["reduccion_pct"]),
         ("2022 VE / ingreso (%)", r22["c10"]["total"]["pct"], _b["ve_pct_total"]),
         ("2022 regresividad D1/D10", r22["c10"]["regresividad"], _b["regresividad"]),
         ("2022 reducción del Gini (%)", r22["g"]["reduccion_pct"], _b["gini_ing_cor"]["reduccion_pct"])],
        {k: 0.011 for k in ("2014 VE / ingreso (%)", "2014 regresividad D1/D10", "2014 reducción del Gini (%)",
                            "2022 VE / ingreso (%)", "2022 regresividad D1/D10", "2022 reducción del Gini (%)")})
